In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(f"Logits: {logits}")
pred_probab = nn.Softmax(dim=1)(logits)
print(f"Predicted probabilities: {pred_probab}")
y_pred = pred_probab.argmax(1) # argmax(1) 的作用是沿维度 1 找出最大值的索引，即获取预测的类别。
print(f"Predicted class: {y_pred}")

Logits: tensor([[-0.0253,  0.0455,  0.0247,  0.0060, -0.0096, -0.1119, -0.0509, -0.0684,
          0.0366,  0.0901]], device='cuda:0', grad_fn=<AddmmBackward0>)
Predicted probabilities: tensor([[0.0980, 0.1051, 0.1030, 0.1011, 0.0995, 0.0898, 0.0955, 0.0938, 0.1042,
         0.1099]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([9], device='cuda:0')


In [7]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [11]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [12]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.3735, -0.1219, -0.4493,  0.2141,  0.1715,  0.0789, -0.1947,  0.0150,
          0.1769, -0.1439,  0.1692, -0.4921, -0.0918,  0.0182,  0.1827, -0.0948,
          0.0132,  0.0761, -0.0491, -0.1449],
        [-0.1103, -0.3818, -0.2569,  0.3054,  0.2905,  0.0635, -0.0274, -0.2682,
          0.3417,  0.4522, -0.1540, -0.2660,  0.0637, -0.1447,  0.2429, -0.0078,
          0.2256,  0.2210, -0.0583, -0.2473],
        [ 0.0781, -0.6310, -0.4826,  0.0250,  0.4497, -0.0573, -0.1443,  0.0571,
          0.5701,  0.3333,  0.0994, -0.2941,  0.3221, -0.2192,  0.2934,  0.2646,
         -0.0068, -0.0329, -0.0301, -0.0412]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.3735, 0.0000, 0.0000, 0.2141, 0.1715, 0.0789, 0.0000, 0.0150, 0.1769,
         0.0000, 0.1692, 0.0000, 0.0000, 0.0182, 0.1827, 0.0000, 0.0132, 0.0761,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3054, 0.2905, 0.0635, 0.0000, 0.0000, 0.3417,
         0.4522, 0.0000, 0.0000, 0.0637, 0.0000, 0.24

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

tensor([[ 0.0711,  0.1050,  0.1485, -0.0979, -0.0870, -0.2713,  0.1198,  0.0983,
          0.1573, -0.0268],
        [ 0.0931,  0.2265,  0.2285, -0.1610, -0.3334, -0.3967,  0.1977,  0.1122,
          0.0819, -0.1091],
        [ 0.0528,  0.1382,  0.1604, -0.0618, -0.2618, -0.2614,  0.2400,  0.1688,
          0.1160, -0.0729]], grad_fn=<AddmmBackward0>)

In [16]:
softmax = nn.Softmax(dim=-1)
pred_probab = softmax(logits)
print(f"pred_probab: {pred_probab}")

y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

pred_probab: tensor([[0.1042, 0.1078, 0.1126, 0.0880, 0.0890, 0.0740, 0.1094, 0.1071, 0.1136,
         0.0945],
        [0.1079, 0.1233, 0.1236, 0.0837, 0.0705, 0.0661, 0.1198, 0.1100, 0.1067,
         0.0882],
        [0.1017, 0.1108, 0.1133, 0.0907, 0.0743, 0.0743, 0.1227, 0.1142, 0.1084,
         0.0897]], grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([8, 2, 6])


In [19]:
# 打印模型架构
print(f"model architecture: {model}")

for name,param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} ")

model architecture: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) 
Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) 
Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) 
Layer: linear_relu_stack.2.bias | Size: torch.Size([512]) 
Layer: linear_relu_stack.4.weight | Size: torch.Size([10, 512]) 
Layer: linear_relu_stack.4.bias | Size: torch.Size([10]) 
